In [27]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test_x.csv")

TARGET = "bilissel_performans_skoru"

X = train.drop(columns=["id", TARGET])
y = train[TARGET]

test_ids = test["id"]
X_test = test.drop(columns=["id"])

print("Train:", X.shape)
print("Test:", X_test.shape)

Train: (56000, 22)
Test: (24000, 22)


In [28]:
def prepare_features(X, X_test):
    X_fe = X.copy()
    X_test_fe = X_test.copy()

    log_cols = [
        "uyku_oncesi_kafein_mg",
        "uyku_oncesi_ekran_suresi_dk"
    ]

    for col in log_cols:
        if col in X_fe.columns:
            X_fe[col] = np.log1p(X_fe[col])
            X_test_fe[col] = np.log1p(X_test_fe[col])

    def add_features(df):
        df = df.copy()

        if "stres_skoru" in df.columns and "gunluk_calisma_saati" in df.columns:
            df["stres_calisma_yuku"] = df["stres_skoru"] * df["gunluk_calisma_saati"]

        if "rem_yuzdesi" in df.columns and "derin_uyku_yuzdesi" in df.columns:
            df["uyku_kalitesi_orani"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]

        if "gecelik_uyanma_sayisi" in df.columns and "uykuya_dalma_suresi_dk" in df.columns:
            df["uyku_bozulma_skoru"] = df["gecelik_uyanma_sayisi"] + df["uykuya_dalma_suresi_dk"]

        if "uyku_oncesi_ekran_suresi_dk" in df.columns and "uyku_oncesi_kafein_mg" in df.columns:
            df["dijital_kafein_yuku"] = df["uyku_oncesi_ekran_suresi_dk"] * df["uyku_oncesi_kafein_mg"]

        if "gunluk_adim_sayisi" in df.columns and "stres_skoru" in df.columns:
            df["aktivite_stres_orani"] = df["gunluk_adim_sayisi"] / (df["stres_skoru"] + 1)

        return df

    X_fe = add_features(X_fe)
    X_test_fe = add_features(X_test_fe)

    return X_fe, X_test_fe


X_fe, X_test_fe = prepare_features(X, X_test)

print("X_fe:", X_fe.shape)
print("X_test_fe:", X_test_fe.shape)

X_fe: (56000, 27)
X_test_fe: (24000, 27)


In [29]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
from catboost import CatBoostRegressor

def get_catboost_oof(X_fe, y, X_test_fe, cv):
    X_cb = X_fe.copy()
    X_test_cb = X_test_fe.copy()

    cat_cols = X_cb.select_dtypes(include="object").columns.tolist()

    for col in cat_cols:
        X_cb[col] = X_cb[col].fillna("Bilinmiyor").astype(str)
        X_test_cb[col] = X_test_cb[col].fillna("Bilinmiyor").astype(str)

    cat_features_idx = [X_cb.columns.get_loc(col) for col in cat_cols]

    oof_preds = np.zeros(len(X_cb))
    test_preds = np.zeros(len(X_test_cb))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_cb), 1):
        X_train_fold = X_cb.iloc[train_idx]
        X_val_fold = X_cb.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            iterations=1500,
            learning_rate=0.035,
            depth=6,
            l2_leaf_reg=5,
            random_seed=42,
            verbose=200,
            early_stopping_rounds=100
        )

        model.fit(
            X_train_fold,
            y_train_fold,
            cat_features=cat_features_idx,
            eval_set=(X_val_fold, y_val_fold),
            use_best_model=True
        )

        val_pred = model.predict(X_val_fold)
        test_pred = model.predict(X_test_cb)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"CatBoost Fold {fold} RMSE:", fold_rmse)

    print("CatBoost Fold RMSE:", fold_scores)
    print("CatBoost Mean RMSE:", np.mean(fold_scores))
    print("CatBoost OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores


oof_cat, test_cat, scores_cat = get_catboost_oof(X_fe, y, X_test_fe, cv)

In [ ]:
def get_hgb_oof(X_fe, y, X_test_fe, cv):
    num_cols = X_fe.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_fe.select_dtypes(include="object").columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ]
    )

    oof_preds = np.zeros(len(X_fe))
    test_preds = np.zeros(len(X_test_fe))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_fe), 1):
        X_train_fold = X_fe.iloc[train_idx]
        X_val_fold = X_fe.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = HistGradientBoostingRegressor(
            max_iter=700,
            learning_rate=0.04,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=0.1,
            random_state=42
        )

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)

        val_pred = pipeline.predict(X_val_fold)
        test_pred = pipeline.predict(X_test_fe)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"HGB Fold {fold} RMSE:", fold_rmse)

    print("HGB Fold RMSE:", fold_scores)
    print("HGB Mean RMSE:", np.mean(fold_scores))
    print("HGB OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores


oof_hgb, test_hgb, scores_hgb = get_hgb_oof(X_fe, y, X_test_fe, cv)

HGB Fold 1 RMSE: 1.228499560987444
HGB Fold 2 RMSE: 1.2290440992060874
HGB Fold 3 RMSE: 1.2159707118361172
HGB Fold 4 RMSE: 1.2243227403073043
HGB Fold 5 RMSE: 1.2450227212554872
HGB Fold RMSE: [np.float64(1.228499560987444), np.float64(1.2290440992060874), np.float64(1.2159707118361172), np.float64(1.2243227403073043), np.float64(1.2450227212554872)]
HGB Mean RMSE: 1.228571966718488
HGB OOF RMSE: 1.2286084071060954


In [ ]:
from scipy.optimize import minimize

oof_matrix = np.column_stack([
    oof_cat,
    oof_hgb
])

test_matrix = np.column_stack([
    test_cat,
    test_hgb
])

def blend_rmse(weights):
    preds = oof_matrix @ weights
    return rmse(y, preds)

n_models = oof_matrix.shape[1]

initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result = minimize(
    blend_rmse,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights = result.x
best_oof_preds = oof_matrix @ best_weights
best_test_preds = test_matrix @ best_weights

print("Best weights:", best_weights)
print("Blend OOF RMSE:", rmse(y, best_oof_preds))

Best weights: [0.84412931 0.15587069]
Blend OOF RMSE: 1.2164008947511131


# Deney 27 - Public Best Çizgisini Baştan Kurma

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor, Pool


# ============================================================
# EXP 27 - Clean CatBoost Native Pipeline
# ============================================================

EXP_NAME = "exp27_cat_native_clean"
SEEDS = [42, 2024, 3407, 777, 999]
N_SPLITS = 5

TARGET_COL = "Degerlendirme Puani"


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))



print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)



cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

for col in cat_features:
    X[col] = X[col].fillna("missing").astype(str)
    X_test[col] = X_test[col].fillna("missing").astype(str)

print("Categorical features:", cat_features)
print("Number of categorical features:", len(cat_features))

print("Remaining categorical NaN in train:", X[cat_features].isna().sum().sum())
print("Remaining categorical NaN in test:", X_test[cat_features].isna().sum().sum())


cat_oof_list_exp27 = []
cat_test_list_exp27 = []
cat_seed_scores_exp27 = {}


for seed in SEEDS:
    print("=" * 70)
    print(f"Starting seed: {seed}")
    print("=" * 70)

    kf = KFold(
        n_splits=N_SPLITS,cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical features:", cat_features)
print("Number of categorical features:", len(cat_features))
    test_pred = np.zeros(len(X_test))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y), start=1):
        print(f"\nSeed {seed} | Fold {fold}")

        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        train_pool = Pool(
            X_train,
            y_train,
            cat_features=cat_features
        )

        valid_pool = Pool(
            X_valid,
            y_valid,
            cat_features=cat_features
        )

        test_pool = Pool(
            X_test,
            cat_features=cat_features
        )

        model = CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            iterations=5000,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=3,
            random_seed=seed,
            verbose=500,
            early_stopping_rounds=300,
            allow_writing_files=False
        )

        model.fit(
            train_pool,
            eval_set=valid_pool,
            use_best_model=True
        )

        valid_pred = model.predict(valid_pool)
        fold_test_pred = model.predict(test_pool)

        oof_pred[valid_idx] = valid_pred
        test_pred += fold_test_pred / N_SPLITS

        fold_rmse = rmse(y_valid, valid_pred)
        fold_scores.append(fold_rmse)

        print(f"Seed {seed} | Fold {fold} RMSE: {fold_rmse:.5f}")

    seed_rmse = rmse(y, oof_pred)

    cat_oof_list_exp27.append(oof_pred)
    cat_test_list_exp27.append(test_pred)
    cat_seed_scores_exp27[seed] = {
        "fold_scores": fold_scores,
        "seed_rmse": seed_rmse
    }

    print(f"\nSeed {seed} OOF RMSE: {seed_rmse:.5f}")


# Güvenlik kontrolü
assert len(cat_oof_list_exp27) > 0, "cat_oof_list_exp27 boş! Model eğitimi çalışmamış."
assert len(cat_test_list_exp27) > 0, "cat_test_list_exp27 boş! Test tahminleri oluşmamış."


# 5 seed ortalaması
oof_cat_exp27_5seed = np.mean(cat_oof_list_exp27, axis=0)
test_cat_exp27_5seed = np.mean(cat_test_list_exp27, axis=0)

final_rmse = rmse(y, oof_cat_exp27_5seed)

print("\n" + "=" * 70)
print(f"{EXP_NAME} Final 5-seed OOF RMSE: {final_rmse:.5f}")
print("=" * 70)


# Kaydet
np.save(f"oof_{EXP_NAME}.npy", oof_cat_exp27_5seed)
np.save(f"test_{EXP_NAME}.npy", test_cat_exp27_5seed)

print(f"Saved: oof_{EXP_NAME}.npy")
print(f"Saved: test_{EXP_NAME}.npy")

X shape: (56000, 23)
y shape: (56000,)
X_test shape: (24000, 23)
Categorical features: ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
Number of categorical features: 7
Remaining categorical NaN in train: 0
Remaining categorical NaN in test: 0
Starting seed: 42

Seed 42 | Fold 1
0:	learn: 2.1962065	test: 2.2069443	best: 2.2069443 (0)	total: 53.9ms	remaining: 4m 29s
500:	learn: 1.1958354	test: 1.2241326	best: 1.2241326 (500)	total: 4.65s	remaining: 41.8s
1000:	learn: 1.1711294	test: 1.2209564	best: 1.2209564 (1000)	total: 8.39s	remaining: 33.5s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 1.220786278
bestIteration = 1079

Shrink model to first 1080 iterations.
Seed 42 | Fold 1 RMSE: 1.22079

Seed 42 | Fold 2
0:	learn: 2.1998516	test: 2.1910527	best: 2.1910527 (0)	total: 7.03ms	remaining: 35.2s
500:	learn: 1.1975040	test: 1.2199261	best: 1.2199241 (498)	total: 3.69s	remaining: 33.2s
1000:	learn: 1.1703787	test: 1.2176194	best: 

In [7]:
exp_results = []

exp_results.append({
    "experiment": "exp27_cat_native_clean",
    "model": "CatBoost",
    "cv": "5-fold x 5-seed",
    "oof_rmse": 1.21469,
    "features": X.shape[1],
    "cat_features": len(cat_features),
    "notes": "Native categorical, categorical missing fixed"
})

pd.DataFrame(exp_results)

,experiment,model,cv,oof_rmse,features,cat_features,notes
0,exp27_cat_native_clean,CatBoost,5-fold x 5-seed,1.21469,23,7,"Native categorical, categorical missing fixed"


In [9]:
test_pred = np.load("test_exp27_cat_native_clean.npy")

submission = pd.DataFrame({
    "id": test["id"],
    "Degerlendirme Puani": test_pred
})

submission.to_csv("submission_exp27_cat_native_clean.csv", index=False)

submission.head()

,id,Degerlendirme Puani
0,1,6.064382
1,2,6.792259
2,3,3.055101
3,4,7.122827
4,5,3.753926


In [10]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [11]:
oof_27 = np.load("oof_exp27_cat_native_clean.npy")
test_27 = np.load("test_exp27_cat_native_clean.npy")

In [12]:
import os

[f for f in os.listdir() if f.endswith(".npy")]

['oof_hgb_v3.npy',
 'test_hgb_v3.npy',
 'test_lgbm_v3.npy',
 'oof_exp27_cat_native_clean.npy',
 'test_exp27_cat_native_clean.npy',
 'oof_lgbm_v3.npy']

In [ ]:
# ============================================================
# DENEY 20E - TEK HÜCRE CLEAN RECREATE
# CatBoost v3 5-seed + HGB v3 + LightGBM v3 optimized blend
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor, Pool


# ------------------------------------------------------------
# Metric
# ------------------------------------------------------------

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

assert "X" in globals(), "X değişkeni yok. Önce feature matrix X oluşturulmalı."
assert "X_test" in globals(), "X_test değişkeni yok. Önce test feature matrix X_test oluşturulmalı."
assert "y" in globals(), "y değişkeni yok. Önce target y oluşturulmalı."

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)
print("y shape:", y.shape)


# ------------------------------------------------------------
# Load HGB and LGBM OOF/Test predictions
# ------------------------------------------------------------

if "oof_hgb_v3" not in globals():
    assert os.path.exists("oof_hgb_v3.npy"), "oof_hgb_v3.npy bulunamadı."
    oof_hgb_v3 = np.load("oof_hgb_v3.npy")

if "test_hgb_v3" not in globals():
    assert os.path.exists("test_hgb_v3.npy"), "test_hgb_v3.npy bulunamadı."
    test_hgb_v3 = np.load("test_hgb_v3.npy")

if "oof_lgbm_v3" not in globals():
    assert os.path.exists("oof_lgbm_v3.npy"), "oof_lgbm_v3.npy bulunamadı."
    oof_lgbm_v3 = np.load("oof_lgbm_v3.npy")

if "test_lgbm_v3" not in globals():
    assert os.path.exists("test_lgbm_v3.npy"), "test_lgbm_v3.npy bulunamadı."
    test_lgbm_v3 = np.load("test_lgbm_v3.npy")


print("\nLoaded base models:")
print("HGB OOF:", oof_hgb_v3.shape, "| Test:", test_hgb_v3.shape, "| RMSE:", rmse(y, oof_hgb_v3))
print("LGBM OOF:", oof_lgbm_v3.shape, "| Test:", test_lgbm_v3.shape, "| RMSE:", rmse(y, oof_lgbm_v3))



cat_oof_file = "oof_cat_seed_ensemble_v3_5.npy"
cat_test_file = "test_cat_seed_ensemble_v3_5.npy"

if "oof_cat_seed_ensemble_v3_5" in globals() and "test_cat_seed_ensemble_v3_5" in globals():
    print("\nUsing existing CatBoost v3 5-seed variables from memory.")

elif os.path.exists(cat_oof_file) and os.path.exists(cat_test_file):
    print("\nLoading CatBoost v3 5-seed predictions from npy files.")
    oof_cat_seed_ensemble_v3_5 = np.load(cat_oof_file)
    test_cat_seed_ensemble_v3_5 = np.load(cat_test_file)

else:
    print("\nCatBoost v3 5-seed predictions not found. Training from scratch...")

    SEEDS = [42, 2024, 3407, 777, 999]
    N_SPLITS = 5


    X_cb = X.copy()
    X_test_cb = X_test.copy()

    cat_features = X_cb.select_dtypes(include=["object", "category"]).columns.tolist()

    for col in cat_features:
        X_cb[col] = X_cb[col].fillna("missing").astype(str)
        X_test_cb[col] = X_test_cb[col].fillna("missing").astype(str)

    print("Categorical features:", cat_features)
    print("Number of categorical features:", len(cat_features))
    print("Remaining categorical NaN train:", X_cb[cat_features].isna().sum().sum())
    print("Remaining categorical NaN test:", X_test_cb[cat_features].isna().sum().sum())

    cat_oof_list_v3_5 = []
    cat_test_list_v3_5 = []
    cat_seed_scores_v3_5 = {}

    for seed in SEEDS:
        print("\n" + "=" * 70)
        print(f"CatBoost v3 5-seed training | Seed: {seed}")
        print("=" * 70)

        kf = KFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=seed
        )

        oof_pred = np.zeros(len(X_cb))
        test_pred = np.zeros(len(X_test_cb))
        fold_scores = []

        for fold, (train_idx, valid_idx) in enumerate(kf.split(X_cb, y), start=1):
            print(f"\nSeed {seed} | Fold {fold}")

            X_train = X_cb.iloc[train_idx]
            y_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]

            X_valid = X_cb.iloc[valid_idx]
            y_valid = y.iloc[valid_idx] if hasattr(y, "iloc") else y[valid_idx]

            train_pool = Pool(
                X_train,
                y_train,
                cat_features=cat_features
            )

            valid_pool = Pool(
                X_valid,
                y_valid,
                cat_features=cat_features
            )

            test_pool = Pool(
                X_test_cb,
                cat_features=cat_features
            )

            model = CatBoostRegressor(
                loss_function="RMSE",
                eval_metric="RMSE",
                iterations=5000,
                learning_rate=0.03,
                depth=6,
                l2_leaf_reg=3,
                random_seed=seed,
                verbose=500,
                early_stopping_rounds=300,
                allow_writing_files=False
            )

            model.fit(
                train_pool,
                eval_set=valid_pool,
                use_best_model=True
            )

            valid_pred = model.predict(valid_pool)
            fold_test_pred = model.predict(test_pool)

            oof_pred[valid_idx] = valid_pred
            test_pred += fold_test_pred / N_SPLITS

            fold_rmse = rmse(y_valid, valid_pred)
            fold_scores.append(fold_rmse)

            print(f"Seed {seed} | Fold {fold} RMSE: {fold_rmse:.6f}")

        seed_rmse = rmse(y, oof_pred)

        cat_oof_list_v3_5.append(oof_pred)
        cat_test_list_v3_5.append(test_pred)

        cat_seed_scores_v3_5[seed] = {
            "fold_scores": fold_scores,
            "seed_rmse": seed_rmse
        }

        print(f"\nSeed {seed} OOF RMSE: {seed_rmse:.6f}")

    assert len(cat_oof_list_v3_5) > 0, "CatBoost OOF listesi boş kaldı."
    assert len(cat_test_list_v3_5) > 0, "CatBoost test listesi boş kaldı."

    oof_cat_seed_ensemble_v3_5 = np.mean(cat_oof_list_v3_5, axis=0)
    test_cat_seed_ensemble_v3_5 = np.mean(cat_test_list_v3_5, axis=0)

    np.save(cat_oof_file, oof_cat_seed_ensemble_v3_5)
    np.save(cat_test_file, test_cat_seed_ensemble_v3_5)

    print("\nSaved:", cat_oof_file)
    print("Saved:", cat_test_file)


print("\nCatBoost v3 5-seed OOF:", rmse(y, oof_cat_seed_ensemble_v3_5))



assert oof_cat_seed_ensemble_v3_5.shape[0] == len(y), "CatBoost OOF shape hatalı."
assert oof_hgb_v3.shape[0] == len(y), "HGB OOF shape hatalı."
assert oof_lgbm_v3.shape[0] == len(y), "LGBM OOF shape hatalı."

assert test_cat_seed_ensemble_v3_5.shape[0] == len(X_test), "CatBoost test shape hatalı."
assert test_hgb_v3.shape[0] == len(X_test), "HGB test shape hatalı."
assert test_lgbm_v3.shape[0] == len(X_test), "LGBM test shape hatalı."




oof_matrix_v3_5 = np.column_stack([
    oof_cat_seed_ensemble_v3_5,
    oof_hgb_v3,
    oof_lgbm_v3
])

test_matrix_v3_5 = np.column_stack([
    test_cat_seed_ensemble_v3_5,
    test_hgb_v3,
    test_lgbm_v3
])

model_names_v3_5 = [
    "CatBoost v3 5 Seed Ensemble",
    "HGB v3",
    "LightGBM v3"
]


def blend_rmse_v3_5(weights):
    preds = oof_matrix_v3_5 @ weights
    return rmse(y, preds)


n_models = oof_matrix_v3_5.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_v3_5 = minimize(
    blend_rmse_v3_5,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_v3_5 = result_v3_5.x
best_oof_preds_v3_5 = oof_matrix_v3_5 @ best_weights_v3_5
best_test_preds_v3_5 = test_matrix_v3_5 @ best_weights_v3_5


print("\n" + "=" * 70)
print("DENEY 20E v3 5-seed optimized blend")
print("=" * 70)

print("Optimization success:", result_v3_5.success)
print("Best weights:")

for name, weight in zip(model_names_v3_5, best_weights_v3_5):
    print(f"{name}: {weight:.6f}")

print("\nBase model OOF scores:")
print("CatBoost v3 5 Seed Ensemble:", rmse(y, oof_cat_seed_ensemble_v3_5))
print("HGB v3:", rmse(y, oof_hgb_v3))
print("LightGBM v3:", rmse(y, oof_lgbm_v3))

print("\nDeney 20E v3 5-seed blend OOF:", rmse(y, best_oof_preds_v3_5))



np.save("oof_20e_recreated.npy", best_oof_preds_v3_5)
np.save("test_20e_recreated.npy", best_test_preds_v3_5)

print("\nSaved: oof_20e_recreated.npy")
print("Saved: test_20e_recreated.npy")



submission_20e_recreated = pd.DataFrame({
    "id": test["id"],
    "Degerlendirme Puani": best_test_preds_v3_5
})

submission_20e_recreated.to_csv("submission_20e_recreated.csv", index=False)

print("\nSubmission saved: submission_20e_recreated.csv")
print(submission_20e_recreated.head())
print("\nSubmission shape:", submission_20e_recreated.shape)
print("\nMissing values:")
print(submission_20e_recreated.isna().sum())

X shape: (56000, 23)
X_test shape: (24000, 23)
y shape: (56000,)

Loaded base models:
HGB OOF: (56000,) | Test: (24000,) | RMSE: 1.2293893343402866
LGBM OOF: (56000,) | Test: (24000,) | RMSE: 1.2310131834046825

CatBoost v3 5-seed predictions not found. Training from scratch...
Categorical features: ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
Number of categorical features: 7
Remaining categorical NaN train: 0
Remaining categorical NaN test: 0

CatBoost v3 5-seed training | Seed: 42

Seed 42 | Fold 1
0:	learn: 2.1962065	test: 2.2069443	best: 2.2069443 (0)	total: 8.11ms	remaining: 40.5s
500:	learn: 1.1958354	test: 1.2241326	best: 1.2241326 (500)	total: 3.67s	remaining: 33s
1000:	learn: 1.1711294	test: 1.2209564	best: 1.2209564 (1000)	total: 8.16s	remaining: 32.6s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 1.220786278
bestIteration = 1079

Shrink model to first 1080 iterations.
Seed 42 | Fold 1 RMSE: 1.220786

Seed 42 | F

In [21]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


oof_20e = np.load("oof_20e_recreated.npy")
test_20e = np.load("test_20e_recreated.npy")

oof_27 = np.load("oof_exp27_cat_native_clean.npy")
test_27 = np.load("test_exp27_cat_native_clean.npy")


print("20E recreated OOF:", rmse(y, oof_20e))
print("Exp27 OOF:", rmse(y, oof_27))


best_score = 999
best_w = None
blend_results = []

for w27 in np.arange(0, 1.001, 0.001):
    blend_oof = (1 - w27) * oof_20e + w27 * oof_27
    score = rmse(y, blend_oof)

    blend_results.append({
        "w_20e": 1 - w27,
        "w_27": w27,
        "rmse": score
    })

    if score < best_score:
        best_score = score
        best_w = w27

blend_results_df = pd.DataFrame(blend_results).sort_values("rmse")

print("Best weight for Exp27:", best_w)
print("Best weight for 20E:", 1 - best_w)
print("Best blend OOF:", best_score)

blend_results_df.head(20)

20E recreated OOF: 1.214421487375675
Exp27 OOF: 1.2146874339867157
Best weight for Exp27: 0.033
Best weight for 20E: 0.967
Best blend OOF: 1.2144211697595806


,w_20e,w_27,rmse
33,0.967,0.033,1.214421
34,0.966,0.034,1.214421
32,0.968,0.032,1.214421
35,0.965,0.035,1.214421
31,0.969,0.031,1.214421
36,0.964,0.036,1.214421
30,0.970,0.030,1.214421
37,0.963,0.037,1.214421
29,0.971,0.029,1.214421
38,0.962,0.038,1.214421


In [22]:
best_w_27 = 0.033
best_w_20e = 0.967

oof_20e = np.load("oof_20e_recreated.npy")
test_20e = np.load("test_20e_recreated.npy")

oof_27 = np.load("oof_exp27_cat_native_clean.npy")
test_27 = np.load("test_exp27_cat_native_clean.npy")

final_oof_blend = best_w_20e * oof_20e + best_w_27 * oof_27
final_test_blend = best_w_20e * test_20e + best_w_27 * test_27

print("Final blend OOF:", rmse(y, final_oof_blend))

Final blend OOF: 1.2144211697595806


In [25]:
submission_final_blend = pd.DataFrame({
    "id": test["id"],
    "Degerlendirme Puani": final_test_blend
})

submission_final_blend.to_csv("submission_20e_recreated_exp27_blend.csv", index=False)

submission_final_blend.head()

,id,Degerlendirme Puani
0,1,6.036028
1,2,6.776145
2,3,3.072168
3,4,7.136535
4,5,3.750549


In [26]:
submission_final_blend = pd.DataFrame({
    "id": test["id"],
    "bilissel_performans_skoru": final_test_blend
})

submission_final_blend.to_csv("submission_20e_recreated_exp27_blend_fixed.csv", index=False)

submission_final_blend.head()

,id,bilissel_performans_skoru
0,1,6.036028
1,2,6.776145
2,3,3.072168
3,4,7.136535
4,5,3.750549


In [27]:
print(submission_final_blend.shape)
print(submission_final_blend.isna().sum())
print(submission_final_blend.head())

(24000, 2)
id                           0
bilissel_performans_skoru    0
dtype: int64
   id  bilissel_performans_skoru
0   1                   6.036028
1   2                   6.776145
2   3                   3.072168
3   4                   7.136535
4   5                   3.750549


In [28]:
exp_results.append({
    "experiment": "20E_recreated_exp27_blend",
    "model": "20E recreated + Exp27",
    "local_oof_rmse": 1.2144211697595806,
    "public_lb": 1.20268,
    "weights": "20E=0.967, Exp27=0.033",
    "submission": "submission_20e_recreated_exp27_blend_fixed.csv",
    "notes": "Improved public score from 1.20273 to 1.20268"
})

pd.DataFrame(exp_results)

,experiment,model,cv,oof_rmse,features,cat_features,notes,local_oof_rmse,public_lb,weights,submission
0,exp27_cat_native_clean,CatBoost,5-fold x 5-seed,1.21469,23.0,7.0,"Native categorical, categorical missing fixed",NaN,NaN,NaN,NaN
1,20E_recreated_exp27_blend,20E recreated + Exp27,NaN,NaN,NaN,NaN,Improved public score from 1.20273 to 1.20268,1.214421,1.20268,"20E=0.967, Exp27=0.033",submission_20e_recreated_exp27_blend_fixed.csv


In [ ]:
# ============================================================
# DENEY 28
# CatBoost 5-seed + Feature Engineering + Best Blend Comparison
# ============================================================

import os
import numpy as np
import pandas as pd

from itertools import combinations
from scipy.optimize import minimize
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor, Pool




def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))



assert "X" in globals(), "X yok."
assert "X_test" in globals(), "X_test yok."
assert "y" in globals(), "y yok."
assert "test" in globals(), "test yok."

print("Original X:", X.shape)
print("Original X_test:", X_test.shape)
print("y:", y.shape)



def safe_div(a, b):
    return a / (b.replace(0, np.nan) + 1e-6)


def find_col(columns, keywords):
    """
    Kolon isimlerinde verilen keyword'leri arar.
    Örn: keywords=["uyku", "süre"] -> uyku_suresi gibi kolonları yakalamaya çalışır.
    """
    cols_lower = {c: c.lower() for c in columns}

    for original_col, lower_col in cols_lower.items():
        ok = True
        for kw in keywords:
            if kw.lower() not in lower_col:
                ok = False
                break
        if ok:
            return original_col

    return None


def add_if_exists_ratio(X_train_fe, X_test_fe, numerator_col, denominator_col, new_col):
    if numerator_col is not None and denominator_col is not None:
        X_train_fe[new_col] = safe_div(X_train_fe[numerator_col], X_train_fe[denominator_col])
        X_test_fe[new_col] = safe_div(X_test_fe[numerator_col], X_test_fe[denominator_col])
        print("Added ratio:", new_col)


def add_if_exists_interaction(X_train_fe, X_test_fe, col1, col2, new_col):
    if col1 is not None and col2 is not None:
        X_train_fe[new_col] = X_train_fe[col1] * X_train_fe[col2]
        X_test_fe[new_col] = X_test_fe[col1] * X_test_fe[col2]
        print("Added interaction:", new_col)



def make_exp28_features(X_base, X_test_base):
    X_fe = X_base.copy()
    X_test_fe = X_test_base.copy()

    for df in [X_fe, X_test_fe]:
        if "id" in df.columns:
            df.drop(columns=["id"], inplace=True)

    cat_cols = X_fe.select_dtypes(include=["object", "category"]).columns.tolist()

    print("\nOriginal categorical columns:", cat_cols)

    for col in cat_cols:
        X_fe[col] = X_fe[col].fillna("missing").astype(str)
        X_test_fe[col] = X_test_fe[col].fillna("missing").astype(str)


    preferred_cat_pairs = [
        ("kronotip", "gun_tipi"),
        ("mevsim", "kronotip"),
        ("meslek", "gun_tipi"),
        ("ruh_sagligi_durumu", "gun_tipi"),
        ("cinsiyet", "meslek"),
        ("ulke", "meslek"),
        ("mevsim", "gun_tipi"),
        ("ruh_sagligi_durumu", "kronotip"),
    ]

    for c1, c2 in preferred_cat_pairs:
        if c1 in X_fe.columns and c2 in X_fe.columns:
            new_col = f"{c1}__{c2}"
            X_fe[new_col] = X_fe[c1].astype(str) + "__" + X_fe[c2].astype(str)
            X_test_fe[new_col] = X_test_fe[c1].astype(str) + "__" + X_test_fe[c2].astype(str)
            print("Added cat combo:", new_col)


    num_cols = X_fe.select_dtypes(include=[np.number]).columns.tolist()

    print("\nOriginal numeric columns:", num_cols)

    X_fe["num_missing_count"] = X_fe[num_cols].isna().sum(axis=1)
    X_test_fe["num_missing_count"] = X_test_fe[num_cols].isna().sum(axis=1)

    X_fe["num_mean"] = X_fe[num_cols].mean(axis=1)
    X_test_fe["num_mean"] = X_test_fe[num_cols].mean(axis=1)

    X_fe["num_std"] = X_fe[num_cols].std(axis=1)
    X_test_fe["num_std"] = X_test_fe[num_cols].std(axis=1)

    X_fe["num_min"] = X_fe[num_cols].min(axis=1)
    X_test_fe["num_min"] = X_test_fe[num_cols].min(axis=1)

    X_fe["num_max"] = X_fe[num_cols].max(axis=1)
    X_test_fe["num_max"] = X_test_fe[num_cols].max(axis=1)

    X_fe["num_range"] = X_fe["num_max"] - X_fe["num_min"]
    X_test_fe["num_range"] = X_test_fe["num_max"] - X_test_fe["num_min"]


    columns = X_fe.columns.tolist()

    sleep_col = find_col(columns, ["uyku"])
    screen_col = find_col(columns, ["ekran"])
    physical_col = find_col(columns, ["fizik"])
    stress_col = find_col(columns, ["stres"])
    age_col = find_col(columns, ["yaş"]) or find_col(columns, ["yas"])
    social_col = find_col(columns, ["sosyal"])
    work_col = find_col(columns, ["çalış"]) or find_col(columns, ["calis"])
    caffeine_col = find_col(columns, ["kafe"]) or find_col(columns, ["kahve"])

    print("\nDetected special columns:")
    print("sleep_col:", sleep_col)
    print("screen_col:", screen_col)
    print("physical_col:", physical_col)
    print("stress_col:", stress_col)
    print("age_col:", age_col)
    print("social_col:", social_col)
    print("work_col:", work_col)
    print("caffeine_col:", caffeine_col)

    add_if_exists_ratio(X_fe, X_test_fe, screen_col, sleep_col, "screen_sleep_ratio")
    add_if_exists_ratio(X_fe, X_test_fe, physical_col, sleep_col, "physical_sleep_ratio")
    add_if_exists_ratio(X_fe, X_test_fe, social_col, screen_col, "social_screen_ratio")
    add_if_exists_ratio(X_fe, X_test_fe, work_col, sleep_col, "work_sleep_ratio")
    add_if_exists_ratio(X_fe, X_test_fe, caffeine_col, sleep_col, "caffeine_sleep_ratio")

    add_if_exists_interaction(X_fe, X_test_fe, stress_col, screen_col, "stress_x_screen")
    add_if_exists_interaction(X_fe, X_test_fe, stress_col, sleep_col, "stress_x_sleep")
    add_if_exists_interaction(X_fe, X_test_fe, stress_col, physical_col, "stress_x_physical")
    add_if_exists_interaction(X_fe, X_test_fe, screen_col, physical_col, "screen_x_physical")
    add_if_exists_interaction(X_fe, X_test_fe, age_col, sleep_col, "age_x_sleep")
    add_if_exists_interaction(X_fe, X_test_fe, age_col, screen_col, "age_x_screen")

    # --------------------------------------------------------
    # 4) Yaş bin feature'ı varsa
    # --------------------------------------------------------

    if age_col is not None:
        X_fe["age_bin"] = pd.cut(
            X_fe[age_col],
            bins=[-np.inf, 18, 25, 35, 45, 60, np.inf],
            labels=["age_0_18", "age_19_25", "age_26_35", "age_36_45", "age_46_60", "age_60_plus"]
        ).astype(str)

        X_test_fe["age_bin"] = pd.cut(
            X_test_fe[age_col],
            bins=[-np.inf, 18, 25, 35, 45, 60, np.inf],
            labels=["age_0_18", "age_19_25", "age_26_35", "age_36_45", "age_46_60", "age_60_plus"]
        ).astype(str)

        print("Added categorical age_bin")

    # --------------------------------------------------------
    # 5) Inf temizliği
    # --------------------------------------------------------

    X_fe = X_fe.replace([np.inf, -np.inf], np.nan)
    X_test_fe = X_test_fe.replace([np.inf, -np.inf], np.nan)

    # Son kategorik kolonlar
    final_cat_cols = X_fe.select_dtypes(include=["object", "category"]).columns.tolist()

    for col in final_cat_cols:
        X_fe[col] = X_fe[col].fillna("missing").astype(str)
        X_test_fe[col] = X_test_fe[col].fillna("missing").astype(str)

    return X_fe, X_test_fe, final_cat_cols


# ------------------------------------------------------------
# Create Exp28 features
# ------------------------------------------------------------

X28, X_test28, cat_features28 = make_exp28_features(X, X_test)

print("\nExp28 X shape:", X28.shape)
print("Exp28 X_test shape:", X_test28.shape)
print("Exp28 categorical feature count:", len(cat_features28))
print("Exp28 categorical features:", cat_features28)
print("Remaining categorical NaN train:", X28[cat_features28].isna().sum().sum())
print("Remaining categorical NaN test:", X_test28[cat_features28].isna().sum().sum())


# ------------------------------------------------------------
# Train CatBoost 5-seed on Exp28 features
# ------------------------------------------------------------

EXP_NAME = "exp28_cat_fe_clean"
SEEDS = [42, 2024, 3407, 777, 999]
N_SPLITS = 5

cat_oof_list_exp28 = []
cat_test_list_exp28 = []
cat_seed_scores_exp28 = {}

for seed in SEEDS:
    print("\n" + "=" * 70)
    print(f"{EXP_NAME} | Seed: {seed}")
    print("=" * 70)

    kf = KFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    oof_pred = np.zeros(len(X28))
    test_pred = np.zeros(len(X_test28))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X28, y), start=1):
        print(f"\nSeed {seed} | Fold {fold}")

        X_train = X28.iloc[train_idx]
        y_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]

        X_valid = X28.iloc[valid_idx]
        y_valid = y.iloc[valid_idx] if hasattr(y, "iloc") else y[valid_idx]

        train_pool = Pool(
            X_train,
            y_train,
            cat_features=cat_features28
        )

        valid_pool = Pool(
            X_valid,
            y_valid,
            cat_features=cat_features28
        )

        test_pool = Pool(
            X_test28,
            cat_features=cat_features28
        )

        model = CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            iterations=6000,
            learning_rate=0.025,
            depth=6,
            l2_leaf_reg=4,
            random_seed=seed,
            verbose=500,
            early_stopping_rounds=350,
            allow_writing_files=False
        )

        model.fit(
            train_pool,
            eval_set=valid_pool,
            use_best_model=True
        )

        valid_pred = model.predict(valid_pool)
        fold_test_pred = model.predict(test_pool)

        oof_pred[valid_idx] = valid_pred
        test_pred += fold_test_pred / N_SPLITS

        fold_rmse = rmse(y_valid, valid_pred)
        fold_scores.append(fold_rmse)

        print(f"Seed {seed} | Fold {fold} RMSE: {fold_rmse:.6f}")

    seed_rmse = rmse(y, oof_pred)

    cat_oof_list_exp28.append(oof_pred)
    cat_test_list_exp28.append(test_pred)

    cat_seed_scores_exp28[seed] = {
        "fold_scores": fold_scores,
        "seed_rmse": seed_rmse
    }

    print(f"\nSeed {seed} OOF RMSE: {seed_rmse:.6f}")


assert len(cat_oof_list_exp28) > 0, "Exp28 OOF listesi boş kaldı."
assert len(cat_test_list_exp28) > 0, "Exp28 test listesi boş kaldı."

oof_exp28_cat_fe_clean = np.mean(cat_oof_list_exp28, axis=0)
test_exp28_cat_fe_clean = np.mean(cat_test_list_exp28, axis=0)

exp28_score = rmse(y, oof_exp28_cat_fe_clean)

print("\n" + "=" * 70)
print(f"{EXP_NAME} OOF RMSE:", exp28_score)
print("=" * 70)

np.save("oof_exp28_cat_fe_clean.npy", oof_exp28_cat_fe_clean)
np.save("test_exp28_cat_fe_clean.npy", test_exp28_cat_fe_clean)

print("Saved: oof_exp28_cat_fe_clean.npy")
print("Saved: test_exp28_cat_fe_clean.npy")


# ------------------------------------------------------------
# Compare with current best
# Current best = 20E recreated + Exp27 blend
# ------------------------------------------------------------

oof_20e = np.load("oof_20e_recreated.npy")
test_20e = np.load("test_20e_recreated.npy")

oof_27 = np.load("oof_exp27_cat_native_clean.npy")
test_27 = np.load("test_exp27_cat_native_clean.npy")

current_best_oof = 0.967 * oof_20e + 0.033 * oof_27
current_best_test = 0.967 * test_20e + 0.033 * test_27

current_best_score = rmse(y, current_best_oof)

print("\nCurrent best OOF:", current_best_score)
print("Exp28 OOF:", exp28_score)


# ------------------------------------------------------------
# Blend current best with Exp28
# ------------------------------------------------------------

best_score = 999
best_w_exp28 = None
blend_results_exp28 = []

for w_exp28 in np.arange(0, 1.001, 0.001):
    blend_oof = (1 - w_exp28) * current_best_oof + w_exp28 * oof_exp28_cat_fe_clean
    score = rmse(y, blend_oof)

    blend_results_exp28.append({
        "w_current_best": 1 - w_exp28,
        "w_exp28": w_exp28,
        "rmse": score
    })

    if score < best_score:
        best_score = score
        best_w_exp28 = w_exp28

blend_results_exp28_df = pd.DataFrame(blend_results_exp28).sort_values("rmse")

print("\n" + "=" * 70)
print("Current Best + Exp28 Blend")
print("=" * 70)

print("Best weight for Exp28:", best_w_exp28)
print("Best weight for current best:", 1 - best_w_exp28)
print("Best blend OOF:", best_score)

display(blend_results_exp28_df.head(20))


# ------------------------------------------------------------
# Save final candidate submission
# ------------------------------------------------------------

final_oof_exp28_blend = (1 - best_w_exp28) * current_best_oof + best_w_exp28 * oof_exp28_cat_fe_clean
final_test_exp28_blend = (1 - best_w_exp28) * current_best_test + best_w_exp28 * test_exp28_cat_fe_clean

print("\nFinal Exp28 blend OOF:", rmse(y, final_oof_exp28_blend))

submission_exp28_blend = pd.DataFrame({
    "id": test["id"],
    "bilissel_performans_skoru": final_test_exp28_blend
})

submission_exp28_blend.to_csv("submission_exp28_current_best_blend.csv", index=False)

print("\nSaved: submission_exp28_current_best_blend.csv")
print(submission_exp28_blend.head())
print("\nShape:", submission_exp28_blend.shape)
print("\nMissing:")
print(submission_exp28_blend.isna().sum())
print("\nPrediction describe:")
print(submission_exp28_blend["bilissel_performans_skoru"].describe())

Original X: (56000, 23)
Original X_test: (24000, 23)
y: (56000,)

Original categorical columns: ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
Added cat combo: kronotip__gun_tipi
Added cat combo: mevsim__kronotip
Added cat combo: meslek__gun_tipi
Added cat combo: ruh_sagligi_durumu__gun_tipi
Added cat combo: cinsiyet__meslek
Added cat combo: ulke__meslek
Added cat combo: mevsim__gun_tipi
Added cat combo: ruh_sagligi_durumu__kronotip

Original numeric columns: ['yas', 'vucut_kitle_indeksi', 'rem_yuzdesi', 'derin_uyku_yuzdesi', 'uykuya_dalma_suresi_dk', 'gecelik_uyanma_sayisi', 'uyku_oncesi_kafein_mg', 'uyku_oncesi_ekran_suresi_dk', 'gunluk_adim_sayisi', 'sekerleme_suresi_dk', 'stres_skoru', 'gunluk_calisma_saati', 'dinlenik_nabiz_bpm', 'oda_sicakligi_celsius', 'hafta_sonu_uyku_farki_saat']

Detected special columns:
sleep_col: derin_uyku_yuzdesi
screen_col: uyku_oncesi_ekran_suresi_dk
physical_col: None
stress_col: stres_skoru
age_col: yas
social_

,w_current_best,w_exp28,rmse
658,0.342,0.658,1.213269
657,0.343,0.657,1.213269
659,0.341,0.659,1.213269
656,0.344,0.656,1.213269
660,0.340,0.660,1.213269
655,0.345,0.655,1.213269
661,0.339,0.661,1.213269
654,0.346,0.654,1.213269
662,0.338,0.662,1.213269
653,0.347,0.653,1.213269



Final Exp28 blend OOF: 1.2132691035554823

Saved: submission_exp28_current_best_blend.csv
   id  bilissel_performans_skoru
0   1                   6.001816
1   2                   6.737925
2   3                   3.044147
3   4                   7.158813
4   5                   3.694694

Shape: (24000, 2)

Missing:
id                           0
bilissel_performans_skoru    0
dtype: int64

Prediction describe:
count    24000.000000
mean         5.937791
std          1.864037
min         -0.229105
25%          4.645149
50%          6.045080
75%          7.321425
max         10.761977
Name: bilissel_performans_skoru, dtype: float64


In [30]:
submission_exp28_blend_clipped = submission_exp28_blend.copy()

submission_exp28_blend_clipped["bilissel_performans_skoru"] = (
    submission_exp28_blend_clipped["bilissel_performans_skoru"]
    .clip(0, 10)
)

submission_exp28_blend_clipped.to_csv(
    "submission_exp28_current_best_blend_clipped.csv",
    index=False
)

print(submission_exp28_blend_clipped.shape)
print(submission_exp28_blend_clipped.isna().sum())
print(submission_exp28_blend_clipped["bilissel_performans_skoru"].describe())

(24000, 2)
id                           0
bilissel_performans_skoru    0
dtype: int64
count    24000.000000
mean         5.937440
std          1.863110
min          0.000000
25%          4.645149
50%          6.045080
75%          7.321425
max         10.000000
Name: bilissel_performans_skoru, dtype: float64
